# Attention Mechanism using Query, Key, and Value (QKV) Demonstration In Python


## **Background**

In Natural Language Processing, models need to understand relationships between words in a sentence.
The attention mechanism helps the model focus on the most relevant words (e.g., understanding that “it” refers to “animal”).

It uses three components — **Query (Q), Key (K), and Value (V)**
- where Query represents what a word is looking for
- Key represents what each word offers
- Value carries the actual information.

Attention works by matching Query with Key to decide how much importance (weight) to give to each word’s Value.



## **Objective**

To train a simple attention mechanism so that a specific word (**“it”**) learns to give higher importance (attention) to the correct related word (**“animal”**).



## **Process Flow**

1. **Input Representation**
   Convert each word in the sentence into numerical embeddings.

2. **Create Trainable Weights**
   Initialize Query (Q) and Key (K) weight matrices.

3. **Compute Attention Scores**
   Calculate similarity between words using -> Q × Kᵀ

4. **Convert to Probabilities**
   Apply softmax to get attention weights (importance of each word).

5. **Define Target**
   Set correct relationship -> “it” should attend to “animal”

6. **Calculate Loss**
   Compare predicted attention with target using cross-entropy loss.

7. **Update Weights**
   Adjust Q and K using gradient descent to reduce error.

8. **Repeat (Training Loop)**
   Run multiple epochs to improve learning.

9. **Final Output**
   After training, attention for “it” becomes highly focused on “animal”.



### Import Libraries

In [20]:
import torch
import torch.nn.functional as F
import numpy as np

### Input Sentence

In [21]:
# Sentence
words = ["The", "animal", "didn’t", "cross", "road",
         "because", "it", "was", "tired"]


### Input embeddings


In [22]:
torch.manual_seed(0)

# Create random embeddings for each word in the sentence
# len(words) → number of words
# 4 → embedding size (each word is represented using 4 numbers)
X = torch.rand((len(words), 4))

### Trainable weights


In [23]:
# Initialize Query weight matrix (Wq)
# Size (4,4) → transforms 4-dim embeddings into 4-dim query vectors
# requires_grad=True → tells PyTorch to track gradients for training
Wq = torch.rand((4, 4), requires_grad=True)

# Initialize Key weight matrix (Wk)
# Size (4,4) → transforms embeddings into key vectors
# requires_grad=True → these weights will be updated during training
Wk = torch.rand((4, 4), requires_grad=True)

### ---------- BEFORE TRAINING ----------


In [24]:
# Create Query (Q) by multiplying input embeddings with Wq
# Each word now gets a "query vector" (what it is looking for)
Q = X @ Wq

# Create Key (K) by multiplying embeddings with Wk
# Each word now gets a "key vector" (what it offers)
K = X @ Wk

# Get dimension of key vectors (used for scaling)
d_k = K.shape[-1]

# Compute attention scores
# Q @ K.T → compares each word with every other word
# Divide by sqrt(d_k) → scaling to avoid very large values
scores = Q @ K.T / np.sqrt(d_k)

# Convert scores into probabilities using softmax
# Each row sums to 1 → gives attention distribution
attention_before = F.softmax(scores, dim=-1)

# Define target: word "it" should focus on "animal"
# Index 1 corresponds to "animal"
target_index = torch.tensor([1])

# Set learning rate → controls how fast weights update
learning_rate = 0.5

# Number of training iterations
epochs = 200

### ---------- TRAINING ----------


In [25]:
# Training loop → repeat learning for multiple epochs
for epoch in range(epochs):

    # Step 1: Compute Query and Key again using updated weights
    Q = X @ Wq
    K = X @ Wk

    # Step 2: Compute attention scores (similarity between words)
    scores = Q @ K.T / np.sqrt(d_k)

    # Step 3: Convert scores to probabilities (attention weights)
    attention_weights = F.softmax(scores, dim=-1)

    # Step 4: Take attention of the word "it"
    # Index 6 → position of "it" in sentence
    # unsqueeze(0) → converts to correct shape for loss function
    predicted = attention_weights[6].unsqueeze(0)

    # Step 5: Compute loss
    # Compare predicted attention with correct target ("animal")
    loss = F.cross_entropy(predicted, target_index)

    # Step 6: Backpropagation
    # Compute gradients (how much to change weights)
    loss.backward()

    # Step 7: Update weights manually using gradient descent
    with torch.no_grad():
        Wq -= learning_rate * Wq.grad
        Wk -= learning_rate * Wk.grad

    # Step 8: Reset gradients to zero before next iteration
    Wq.grad.zero_()
    Wk.grad.zero_()

### ---------- AFTER TRAINING ----------


In [26]:
# Recompute Query and Key using the UPDATED (trained) weights
Q = X @ Wq
K = X @ Wk

# Compute attention scores again
# Now these scores reflect learned relationships between words
scores = Q @ K.T / np.sqrt(d_k)

# Convert scores into probabilities (attention weights)
# This shows where each word is focusing AFTER training
attention_after = F.softmax(scores, dim=-1)

### ---------- PRINT COMPARISON ----------


In [27]:

print("\n🔹 Attention of 'it' BEFORE training:\n")
for i, word in enumerate(words):
    print(f"{word} : {attention_before[6][i].item():.4f}")

print("\n🔹 Attention of 'it' AFTER training:\n")
for i, word in enumerate(words):
    print(f"{word} : {attention_after[6][i].item():.4f}")


🔹 Attention of 'it' BEFORE training:

The : 0.0642
animal : 0.1662
didn’t : 0.0997
cross : 0.0539
road : 0.1030
because : 0.2548
it : 0.1117
was : 0.1122
tired : 0.0343

🔹 Attention of 'it' AFTER training:

The : 0.0000
animal : 0.9509
didn’t : 0.0000
cross : 0.0001
road : 0.0000
because : 0.0174
it : 0.0037
was : 0.0280
tired : 0.0000


### Conclusion
After training, the model successfully learns the correct relationship, with “it” strongly attending to “animal” instead of distributing attention randomly.